# 1. Fundamental matrix definition

The fundamental matrix $F$ is a $3 \times 3$ matrix that describes the epipolar geometry between two images of the same scene.

- Fundamental matrix $F$: Describes epipolar geometry directly in pixel coordinates. It does not require camera calibration to estimate.
- Essential matrix $E$: Describes epipolar geometry using normalized camera coordinates. It requires camera calibration.

The fundamental matrix allows us to calculate the epipolar line:
    $$\boxed{\mathbf l_2=\mathbf F\mathbf p_1}$$

where:
- $ p_1=[u_1,v_1,1]^T$ is a point in the first image, expressed in homogeneous pixel coordinates.
- $F$ is the fundamental matrix.
- $l_2=[a,b,c]^T$ represents an epipolar line in the second image.

The line equation is:
    $$au_2+bv_2+c=0$$

Consequently, the corresponding point $p_2$ must satisfy:
    $$ p_2^T F p_1=0$$

This is the fundamental matrix constraint.

In real images, because of feature localization errors and noise, this equation is usually only approximately satisfied.

# 2. Fundamental matrix estimation

We can estimate $\mathbf F$ directly from corresponding feature points in two images.

One classical method is the 8-point algorithm.

Suppose we have eight corresponding point pairs:
$$
(\mathbf p_1^{(i)},\mathbf p_2^{(i)}),\quad i=1,\ldots,8
$$

For each pair:
$$
\mathbf p_2^T\mathbf F\mathbf p_1=0
$$

Expanding this equation:
$$
\begin{aligned}
&u_2u_1f_{11}+u_2v_1f_{12}+u_2f_{13}\\
&+v_2u_1f_{21}+v_2v_1f_{22}+v_2f_{23}\\
&+u_1f_{31}+v_1f_{32}+f_{33}=0
\end{aligned}
$$

This equation is linear with respect to the nine unknown elements of $\mathbf F$.

For eight correspondences, we construct a system:
$$
\mathbf A\mathbf f=0
$$

where $\mathbf A$ is an $8\times9$ matrix, and $\mathbf f$ contains the nine elements of $\mathbf F$.

We then:
1. Normalize the image point coordinates for numerical stability.
2. Use SVD to find a solution for $\mathbf f$.
3. Reshape $\mathbf f$ into a $3\times3$ matrix.
4. Enforce the rank-2 constraint using another SVD.
5. Transform the matrix back to the original pixel coordinate systems (denormalize).

**Note**: notice that there are two separate uses of SVD in the 8-point algorithm:
- First SVD: Applied to $A$, to estimate the elements of $F$.
- Second SVD: Applied to the estimated $F$, to enforce its rank-2 constraint.

In practice, we usually combine fundamental matrix estimation with RANSAC to reject incorrect feature matches.

# 3. Rank-2 enforcement

The fundamental matrix $F$ must have rank exactly 2 for a nondegenerate pair of cameras.

The output of the 8-point algorithm is an estimation of matrix $F$, but it does not guarantee that the resulting matrix has rank 2.
Because of measurement noise, the estimated matrix will generally have rank 3.

We use SVD to correct this:
1. Perform SVD on the estimated fundamental matrix

   Suppose we have estimated:
    $$
    \mathbf F=
    \begin{bmatrix}
    f_{11}&f_{12}&f_{13}\\
    f_{21}&f_{22}&f_{23}\\
    f_{31}&f_{32}&f_{33}
    \end{bmatrix}
    $$

    Perform SVD:
    $$
    \mathbf F=\mathbf U\mathbf\Sigma\mathbf V^T
    $$

    where:
    $$
    \mathbf\Sigma=
    \begin{bmatrix}
    \sigma_1&0&0\\
    0&\sigma_2&0\\
    0&0&\sigma_3
    \end{bmatrix}
    $$

    with $\sigma_1\geq\sigma_2\geq\sigma_3\geq0$.

    For example:
    $$
    \mathbf\Sigma=
    \begin{bmatrix}
    12&0&0\\
    0&4&0\\
    0&0&0.02
    \end{bmatrix}
    $$

    Since all three singular values are nonzero, the matrix has rank 3.

2. Set the smallest singular value to zero
    We construct a new diagonal matrix:
    $$
    \boxed{
    \mathbf\Sigma'=
    \begin{bmatrix}
    \sigma_1&0&0\\
    0&\sigma_2&0\\
    0&0&0
    \end{bmatrix}
    }
    $$

    In example:
    $$
    \mathbf\Sigma'=
    \begin{bmatrix}
    12&0&0\\
    0&4&0\\
    0&0&0
    \end{bmatrix}
    $$

    Reconstruct the fundamental matrix:
    $$
    \boxed{\mathbf F'=\mathbf U\mathbf\Sigma'\mathbf V^T}
    $$

    The resulting matrix has rank 2, assuming the first two singular values are nonzero.

## Mathematical justification

According to the Eckart–Young–Mirsky theorem, this operation produces the closest rank-2 matrix to the original $F$ in Frobenius norm:
$$
\mathbf F'
=
\underset{\operatorname{rank}(\mathbf X)\leq2}{\arg\min}
\|\mathbf F-\mathbf X\|_F
$$

In other words, we make the smallest possible correction (under this matrix norm) to satisfy the rank constraint.

## Rank-2 requirement justification

### Geometric explanation

Recall that the fundamental matrix transforms a point in the first image into an epipolar line in the second image:
    $$
    \mathbf l_2=\mathbf F\mathbf p_1
    $$

Now recall the equation expressing that the epipole belongs to every epipolar line:
$$
\mathbf e_2^T\mathbf l_2=0
$$

Substitute $\mathbf l_2=\mathbf F\mathbf p_1$:
$$
\mathbf e_2^T\mathbf F\mathbf p_1=0
$$

For this to hold for all $\mathbf p_1$:
$$
\boxed{\mathbf F^T\mathbf e_2=\mathbf0}
$$

The epipole is a nonzero homogeneous vector. Therefore, $\mathbf F^T$ must have a nontrivial null space.

A full-rank $3\times3$ matrix has only the zero vector in its null space.

Consequently:
$$
\operatorname{rank}(\mathbf F)\leq2
$$

This proves why rank 3 is geometrically impossible.

For two distinct camera centers in a nondegenerate configuration, the rank is exactly 2.

### Algebraic explanation
Recall:
$$
\mathbf F=\mathbf K_2^{-T}[\mathbf t]_\times\mathbf R\mathbf K_1^{-1}
$$

The matrices $\mathbf K_1^{-1}$, $\mathbf K_2^{-T}$, and $\mathbf R$ are invertible, so they do not change the rank.

Therefore:
$$
\operatorname{rank}(\mathbf F)
=
\operatorname{rank}([\mathbf t]_\times)
$$

For nonzero translation $t$, the skew-symmetric matrix
$$
[\mathbf t]_\times=
\begin{bmatrix}
0&-t_z&t_y\\
t_z&0&-t_x\\
-t_y&t_x&0
\end{bmatrix}
$$

has rank 2 because:
$$
[\mathbf t]_\times\mathbf t=0
$$

Its null space contains the nonzero vector $t$, and for $t\ne0$, its null space has dimension exactly 1.

Thus, $\mathbf F$ must have rank 2.

# 4. Null-space

The null space (kernel) of a matrix $A$ is the set of all vectors $x$ that satisfy:
    $$
    \boxed{\mathbf A\mathbf x=\mathbf 0}
    $$

Example:
    $$
    \mathbf A=
    \begin{bmatrix}
    1&2\\
    2&4
    \end{bmatrix}
    $$
    $$
    \mathbf A\mathbf x=\mathbf 0
    $$
    $$
    \begin{cases}
    x_1+2x_2=0\\
    2x_1+4x_2=0
    \end{cases}
    $$
    The second equation is redundant, so:
    $$
    x_1=-2x_2
    $$
    Therefore, all solutions have the form:
    $$
    \boxed{
    \mathbf x=t
    \begin{bmatrix}
    -2\\1
    \end{bmatrix},\quad t\in\mathbb R
    }
    $$
    The null space is a one-dimensional subspace: a line through the origin. Geometrically, the transformation represented by $A$ collapses the entire line to the origin.

**Rank-nullity Theorem:**
$$
\boxed{\operatorname{rank}(\mathbf A)+\dim(\ker(\mathbf A))=n}
$$

where:
- $n$ is the number of columns of $A$.
- $\ker(A)$ – kernel of $A$ $==$ null space of $A$
- $\dim$ - dimensionality. The number of vectors in a basis for that subspace (how "large" or the number of degrees of freedom the kernel has).

In the example above, $n=2$, rank is 1, and null-space dimension is 1.

## Null spaces of the fundamental matrix

Now consider the fundamental matrix:
$$
\mathbf F\in\mathbb R^{3\times3},\qquad \operatorname{rank}(\mathbf F)=2
$$

From the rank-nullity theorem:
$$
\dim\ker(\mathbf F)=3-2=1
$$

Therefore, there exists a one-dimensional null space containing nonzero vectors.

The same applies to $F^T$, because transposition does not change rank.

We have two important relationships:
$$
\boxed{\mathbf F\mathbf e_1=\mathbf0}
$$

$$
\boxed{\mathbf F^T\mathbf e_2=\mathbf0}
$$

Here:
- $e_1$ is the epipole in the first image.
- $e_2$ is the epipole in the second image.

These are called the right and left null-space relationships of $\mathbf F$.

## Why are epipoles in the null spaces?

Recall that an epipole is the projection of the other camera's center onto an image plane.
All epipolar lines in a given image pass through its epipole.

Let's derive the relationship for the second image.

Step 1. A point $\mathbf p_1$ in the first image determines an epipolar line in the second image:
$$
\mathbf l_2=\mathbf F\mathbf p_1
$$

Step 2. Every epipolar line in the second image passes through $\mathbf e_2$.

A point $\mathbf e_2$ lies on a line $\mathbf l_2$ if:
$$
\mathbf e_2^T\mathbf l_2=0
$$

Step 3. Substitute the expression for the epipolar line:
$$
\mathbf e_2^T\mathbf F\mathbf p_1=0
$$

This equation must hold for every possible point $\mathbf p_1$.

Therefore:
$$
\mathbf e_2^T\mathbf F=\mathbf0^T
$$

Taking the transpose:
$$
\boxed{\mathbf F^T\mathbf e_2=\mathbf0}
$$

We have proved that the epipole in the second image belongs to the null space of $\mathbf F^T$.

Similarly, we can derive:
$$
\boxed{\mathbf F\mathbf e_1=\mathbf0}
$$

Geometric interpretation: The null spaces identify the two epipoles, which are the common intersection points of the epipolar lines in their respective images.
The Mathematics of Modern Computer Vision


An additional subtlety: when we calculate $\mathbf F\mathbf e_1=\mathbf0$, the result is not a valid epipolar line. The zero vector cannot represent a line in homogeneous coordinates. This corresponds to the geometric ambiguity of the epipolar line associated with the epipole itself.

## How to find the null spaces using SVD?

$$
\mathbf F=\mathbf U\mathbf\Sigma\mathbf V^T
$$

If:
$$
\mathbf\Sigma=\operatorname{diag}(\sigma_1,\sigma_2,0)
$$

The singular vectors corresponding to the zero singular value give us the null spaces:
- Last column of $V$: Right null vector, corresponding to $e_1$.
- Last column of $U$: Left null vector, corresponding to $e_2$.

# 5. Important properties

The rank-2 property means:
$$
\det(\mathbf F)=0
$$

The matrix also has two important null-space relationships:
$$
\mathbf F\mathbf e_1=0
$$

$$
\mathbf F^T\mathbf e_2=0
$$

Here, $\mathbf e_1$ and $\mathbf e_2$ are the epipoles in the two images.

# 6. Relationship between fundamental and essential matrices

Recall the essential matrix equation:
    $$\hat{\mathbf p}_2^T\mathbf E\hat{\mathbf p}_1=0$$

Here, $\hat{\mathbf p}_1$ and $\hat{\mathbf p}_2$ are normalized camera coordinates.

We can convert pixel coordinates to normalized camera coordinates using the intrinsic calibration matrix:
$$
\hat{\mathbf p}_1=\mathbf K_1^{-1}\mathbf p_1
$$

$$
\hat{\mathbf p}_2=\mathbf K_2^{-1}\mathbf p_2
$$

Substituting these expressions into the essential matrix equation:
$$
(\mathbf K_2^{-1}\mathbf p_2)^T
\mathbf E
(\mathbf K_1^{-1}\mathbf p_1)=0
$$

Rearranging:
$$
\mathbf p_2^T
\underbrace{\mathbf K_2^{-T}\mathbf E\mathbf K_1^{-1}}_{\mathbf F}
\mathbf p_1=0
$$

Therefore:
$$
\boxed{\mathbf F=\mathbf K_2^{-T}\mathbf E\mathbf K_1^{-1}}
$$

Since you already know that:
$$
\mathbf E=[\mathbf t]_\times\mathbf R
$$

we obtain:
$$
\boxed{\mathbf F=\mathbf K_2^{-T}[\mathbf t]_\times\mathbf R\mathbf K_1^{-1}}
$$

This gives us an important interpretation: the fundamental matrix combines relative camera geometry with the intrinsic parameters of both cameras.

# 7. Practical example in OpenCV
Suppose you already have feature correspondences from ORB matching:

```python
import cv2
import numpy as np

# Nx2 arrays of corresponding pixel coordinates
points1 = np.array([...], dtype=np.float32)
points2 = np.array([...], dtype=np.float32)

F, mask = cv2.findFundamentalMat(
    points1,
    points2,
    cv2.FM_RANSAC,
    1.0,
    0.99
)

# Keep geometrically consistent matches
inliers1 = points1[mask.ravel() == 1]
inliers2 = points2[mask.ravel() == 1]
```

The resulting $\mathbf F$ can be used to calculate epipolar lines or filter inconsistent matches.

# Parallel image planes

When the two camera image planes are parallel, we can still use the fundamental matrix.

Let's distinguish two cases:
1. Parallel image planes with sideways translation: Epipoles are at infinity, and epipolar lines are parallel.
2. Parallel image planes with forward translation: Epipoles are finite, and epipolar lines converge at them.

## Sideways translation: epipoles at infinity

Imagine two cameras with identical orientation, where the second camera is translated horizontally.

<img src="5_images/sideways_translation.svg" width="500">

In projective geometry, parallel lines intersect at a point at infinity.
This is why homogeneous coordinates are particularly useful.

A finite point has homogeneous coordinates:
$$
\mathbf e=[u,v,1]^T
$$

A point at infinity can be represented as:
$$
\boxed{\mathbf e=[a,b,0]^T}
$$

We cannot convert this into ordinary pixel coordinates by dividing by the third component, because it is zero.

But the fundamental matrix equations remain valid:
$$
\mathbf F\mathbf e_1=0
$$

$$
\mathbf F^T\mathbf e_2=0
$$

The rank-2 requirement still holds.

### Example

Assume two calibrated cameras have:
- Identical orientation: $\mathbf R=\mathbf I$
- Horizontal relative translation: $\mathbf t=[B,0,0]^T$
- Identical intrinsics: $\mathbf K=\operatorname{diag}(f,f,1)$, with principal point at the origin

The essential matrix is:
$$
\mathbf E=[\mathbf t]_\times\mathbf R
=
\begin{bmatrix}
0&0&0\\
0&0&-B\\
0&B&0
\end{bmatrix}
$$

The fundamental matrix is:
$$
\mathbf F=\mathbf K^{-T}\mathbf E\mathbf K^{-1}
=
\begin{bmatrix}
0&0&0\\
0&0&-B/f\\
0&B/f&0
\end{bmatrix}
$$

Since \(\mathbf F\) is defined only up to a nonzero scale, we can simplify it to:
$$
\boxed{
\mathbf F\sim
\begin{bmatrix}
0&0&0\\
0&0&-1\\
0&1&0
\end{bmatrix}
}
$$

Now apply the epipolar constraint:
$$
\mathbf p_2^T\mathbf F\mathbf p_1=0
$$

For $\mathbf p_1=[u_1,v_1,1]^T$ and $\mathbf p_2=[u_2,v_2,1]^T$:
$$
-v_2+v_1=0
$$

Therefore:
$$
\boxed{v_1=v_2}
$$

Corresponding points have the same vertical coordinate. This is exactly why stereo matching can be reduced to searching along horizontal image rows.

Notice that this matrix still has rank 2, even though the epipolar lines are parallel.

Its epipole is:
$$
\mathbf e=[1,0,0]^T
$$

which represents a point at infinity in the horizontal direction.

## Forward translation: epipoles are finite

Suppose the two image planes are still parallel, but the camera translates along its optical axis:
$$
\mathbf t=[0,0,B]^T
$$

Now the epipole is finite, at the principal point (assuming the same intrinsics).

The epipolar lines radiate outward from that point rather than being parallel.

<img src="5_images/forward_translation.jpeg" width="750">